In [ ]:
import wandb

api = wandb.Api()
artifact = api.artifact("gteller-cmu/en-fr-10701/ssm_large_5_best.pt:v145")

os.makedirs("temp", exist_ok=True)
artifact.download("temp/")

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /home/ec2-user/.netrc.
wandb: Downloading large artifact 'ssm_large_5_best.pt:v145', 635.31MB. 2 files...
wandb:   2 of 2 files downloaded.  
Done. 00:00:00.4 (1522.4MB/s)


SSMTranslator(
  (E): Embedding(32000, 768)
  (encoder_layers): ModuleList(
    (0-4): 5 x Mamba2Layer(
      (lin_logA): MultiheadLinear()
      (lin_XBC): MultiheadLinear()
      (conv): Conv1d(2304, 2304, kernel_size=(4,), stride=(1,), groups=2304)
      (lin_gate): MultiheadLinear()
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (lin_out): Linear(in_features=768, out_features=768, bias=True)
    )
    (5): Mamba2Layer(
      (lin_logA): MultiheadLinear()
      (lin_XBC): MultiheadLinear()
      (conv): Conv1d(1536, 1536, kernel_size=(4,), stride=(1,), groups=1536)
    )
  )
  (lin_EDs): ModuleList(
    (0-5): 6 x Linear(in_features=64, out_features=64, bias=True)
  )
  (decoder_layers): ModuleList(
    (0-5): 6 x Mamba2Layer(
      (lin_logA): MultiheadLinear()
      (lin_XBC): MultiheadLinear()
      (conv): Conv1d(2304, 2304, kernel_size=(4,), stride=(1,), groups=2304)
      (lin_gate): MultiheadLinear()
      (norm): LayerNorm((768,), eps=1e-05, elemen

In [1]:
import os
from ssm import *
import yaml

model = SSMTranslator(config=SSMTranslatorConfig(**yaml.safe_load(open("../configs/ssm_tuning/model.yaml", "r"))))
state_dict = { x[len("module."):]: v for x, v in torch.load("../temp/best_model.pt").items() }
model.load_state_dict(state_dict)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

SSMTranslator(
  (E): Embedding(32000, 768)
  (encoder_layers): ModuleList(
    (0-4): 5 x Mamba2Layer(
      (lin_logA): MultiheadLinear()
      (lin_XBC): MultiheadLinear()
      (conv): Conv1d(2304, 2304, kernel_size=(4,), stride=(1,), groups=2304)
      (lin_gate): MultiheadLinear()
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (lin_out): Linear(in_features=768, out_features=768, bias=True)
    )
    (5): Mamba2Layer(
      (lin_logA): MultiheadLinear()
      (lin_XBC): MultiheadLinear()
      (conv): Conv1d(1536, 1536, kernel_size=(4,), stride=(1,), groups=1536)
    )
  )
  (lin_EDs): ModuleList(
    (0-5): 6 x Linear(in_features=64, out_features=64, bias=True)
  )
  (decoder_layers): ModuleList(
    (0-5): 6 x Mamba2Layer(
      (lin_logA): MultiheadLinear()
      (lin_XBC): MultiheadLinear()
      (conv): Conv1d(2304, 2304, kernel_size=(4,), stride=(1,), groups=2304)
      (lin_gate): MultiheadLinear()
      (norm): LayerNorm((768,), eps=1e-05, elemen

In [7]:
import sentencepiece as sp

proc_en = sp.SentencePieceProcessor()
proc_fr = sp.SentencePieceProcessor()

proc_en.Load("../vocab/en.model")
proc_fr.Load("../vocab/fr.model")

sentence = "I am a sentence which should be translated from English to French."
en_ids = proc_en.Encode(sentence, add_bos=True, add_eos=True)
en_ids = torch.tensor(en_ids).unsqueeze(0).to(device)

fr_ids = proc_fr.Encode("Je suis une phrase qui devrait être traduite de l'anglais au français.", add_bos=True, add_eos=True)
fr_ids = torch.tensor(fr_ids).unsqueeze(0).to(device)

print(en_ids)
print(fr_ids)

logits1 = model(
    inp_ids=en_ids,
    decode_method="forced",
    forcing_ids=fr_ids[:, :-1],
)

logits2 = model(
    inp_ids=en_ids,
    decode_method="ag",
    max_output_len=64,
)

ids1 = torch.argmax(logits1, dim=-1).squeeze(0).tolist()
ids2 = torch.argmax(logits2, dim=-1).squeeze(0).tolist()
print(ids1)
print(ids2)

print(proc_fr.Decode(ids1))
print(proc_fr.Decode(ids2))

tensor([[    1,    66,   416,     5,  9367,   258,   465,    68, 13237,   199,
          2999,    36,  2263, 31036,     2]], device='cuda:0')
tensor([[    1,  1862,  4630,   127, 13304,   130,  1370,   330, 28625,    11,
             6, 31004,  7951,    52,  2671, 31001,     2]], device='cuda:0')
[1862, 4630, 4, 13304, 130, 1370, 330, 20838, 68, 26, 31008, 7951, 31001, 2671, 31001, 2]
[1862, 4630, 4, 31004, 2220, 286, 26, 3222, 11, 6, 31004, 7951, 31001, 2]
Je suis d phrase qui devrait être rédigée par la’anglais. français.
Je suis d'accord avec la publication de l'anglais.
